<a href="https://colab.research.google.com/github/tashir0605/Cocepts-And-Practice/blob/main/Machine%20Learning/Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install Optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.2 MB/s eta 0:00:00


In [8]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source

# Load the Pima Indian Diabetes dataset (from UCI repository)
# Define the URL pointing to the raw CSV data source
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"

# Define the explicit column names for the dataset since the raw file lacks a header row
columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
    'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

# Load the dataset into a pandas DataFrame using the specified URL and column headers
# Use skiprows=1 to skip the header row in the CSV file itself
df = pd.read_csv(url, names=columns, skiprows=1)

# Display the first 5 rows of the loaded DataFrame to verify the data structure
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [10]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


so coming to objective functions here we are passing the trial object

and then we have defined our search space of the paramneters

    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)





In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score # Return the accuracy score for Optuna to maximize


So, what is the flow of optuna->  

` study=optuna.create_study(direction='maximize')`
this line creates a study, a study means that let's suppose you ahve to try different-different value of max_depth to check on which it is giving the best result trying different value of paramenters called as a study and trying no of different value parameters called as trials.

So this line creates a study where I have different parameters to check and direction is called as maximime means our objective is to get maxima of the score that objective function will produce.

`study.optimize(objective,n_trials=100)`

Now coming to this  study calls optimize function where we are passing objective function and no of trials.


In [12]:
study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=100)

[I 2026-06-14 11:57:27,598] A new study created in memory with name: no-name-abc5d71f-c4c3-4715-bfe3-33672402e052
[I 2026-06-14 11:57:29,531] Trial 0 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 7}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-06-14 11:57:30,582] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 172, 'max_depth': 6, 'min_samples_split': 4}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-06-14 11:57:31,054] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 84, 'max_depth': 7, 'min_samples_split': 5}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-06-14 11:57:31,508] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 75, 'max_depth': 18, 'min_samples_split': 7}. Best is trial 0 with value: 0.7765363128491621.
[I 2026-06-14 11:57:32,053] Trial 4 finished with value: 0.765363128491620

In [13]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75
